# Go2 Track 2 Bonus Project: Colab Starter Notebook

Use this notebook to set up the repo, inspect the interfaces, optionally train or reuse a low-level checkpoint, run single-policy track evaluation, and prepare submission metadata.

It starts from a weak baseline. Your leaderboard submission should train a learned high-level planner for the fixed 5D -> [vx, vy, yaw_rate] interface.


## 1. Configure repository URLs

Leave the default repo URL unless you are working from your own fork. If you rerun setup after editing files, keep `RESET_COURSE_REPO = False` so your local changes are not deleted.


In [ ]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

COURSE_REPO_URL = "https://github.com/jiarao76/Final-Project-Track-2-Bonus-Project.git"
COURSE_REPO_BRANCH = "main"
TEAM_NAME = "Jiarao_Zhang"
RESET_COURSE_REPO = False
BASE_DIR = Path("/content") if Path("/content").exists() else Path("/tmp/go2_track_bonus_colab")
BASE_DIR.mkdir(parents=True, exist_ok=True)
COURSE_REPO_DIR = BASE_DIR / "go2_track_bonus_repo"

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"

PLAYGROUND_DIR = BASE_DIR / "mujoco_playground"
UNITREE_DIR = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        if RESET_COURSE_REPO:
            print(f"+ remove existing course repo at {target_dir}")
            shutil.rmtree(target_dir)
        else:
            print(f"+ reuse existing course repo at {target_dir}")
            return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")


## 2. Install system packages and clone repositories


In [ ]:
import shutil
if shutil.which("ffmpeg") is None:
    if Path("/content").exists():
        run(["apt-get", "update", "-qq"])
        run(["apt-get", "install", "-y", "ffmpeg"])
    else:
        print("[local] ffmpeg is not on PATH; imageio-ffmpeg from requirements is enough for local tests.")
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground || true

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}
%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .
%cd {COURSE_REPO_DIR}

import sys
if str(PLAYGROUND_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_DIR.resolve()))

import jax
import mujoco_playground

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())
print("mujoco_playground imported from:", mujoco_playground.__file__)
expected_playground = str(PLAYGROUND_DIR.resolve())
if expected_playground not in str(Path(mujoco_playground.__file__).resolve()):
    raise RuntimeError(f"Expected mujoco_playground to be imported from {expected_playground}")


## 2b. Write MLP planner and training script

Patches `track_bonus/planner.py` so `StarterTrackPlanner.load()` can load
the new `MLPTrackPlanner` (selected by `planner_type: mlp` in config).
Also writes the CMA-ES training script `train_highlevel_mlp.py`.


In [ ]:
# Patch planner.py with MLP version and write training script
import base64 as _b64
_planner_b64 = 'IiIiSGlnaC1sZXZlbCBwbGFubmVyIGZvciB0aGUgMjAwIG0gdHJhY2sgYm9udXMuCgpTdXBwb3J0cyB0d28gcGxhbm5lciB0eXBlcyBzZWxlY3RlZCB2aWEgdGhlIEpTT04gY29uZmlnIGZpZWxkIGBgcGxhbm5lcl90eXBlYGA6CgogICogYGAic3RhcnRlcl9wZCJgYCDigJMgb3JpZ2luYWwgcHJvcG9ydGlvbmFsLWRlcml2YXRpdmUgYmFzZWxpbmUgKHVuY2hhbmdlZCkuCiAgKiBgYCJtbHAiYGAgICAgICAgIOKAkyBsZWFybmVkIE1MUCBwbGFubmVyIHdpdGggQ01BLUVTLXRyYWluZWQgd2VpZ2h0cy4KClRoZSBldmFsdWF0b3IgZW50cnkgcG9pbnQgaXMgYWx3YXlzOjoKCiAgICBwbGFubmVyID0gU3RhcnRlclRyYWNrUGxhbm5lci5sb2FkKHBhdGhfdG9fY29uZmlnX2pzb24pCiAgICBjbWQgPSBwbGFubmVyLmNvbW1hbmQodHJhY2tfb2JzLCB0KSAgICAgICAgICAjIC0+IG5wLm5kYXJyYXkgc2hhcGUgKDMsKQoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgVW5pb24KCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBnbzJfcGdfZW52LnRyYWNrIGltcG9ydCBTdGFuZGFyZE92YWxUcmFjaywgd3JhcF9hbmdsZQpmcm9tIHRyYWNrX2JvbnVzLmNvbnRyb2xsZXJfaW50ZXJmYWNlIGltcG9ydCBUcmFja0NvbnRyb2xsZXJPYnNlcnZhdGlvbgpmcm9tIHRyYWNrX2JvbnVzLm9mZmljaWFsX3RyYWNrIGltcG9ydCBvZmZpY2lhbF90cmFjawoKIyDilIDilIAgUGh5c2ljYWwgY29tbWFuZCBsaW1pdHMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACl9WWF9NSU46IGZsb2F0ID0gMC4xNSAgICMgbS9zIOKAkyBuZXZlciB3YWxrIHNsb3dlciB0aGFuIHRoaXMKX1ZYX01BWDogZmxvYXQgPSAwLjUwICAgIyBtL3Mg4oCTIHRvcCBmb3J3YXJkIHNwZWVkCl9WWV9MSU06IGZsb2F0ID0gMC4xMCAgICMgbS9zIOKAkyBsYXRlcmFsIGNvcnJlY3Rpb24gbGltaXQKX1lBV19MSU06IGZsb2F0ID0gMC4zMCAgIyByYWQvcyDigJMgeWF3IHJhdGUgbGltaXQKCgojIOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkAojIFN0YXJ0ZXJQbGFubmVyQ29uZmlnICAodW5jaGFuZ2VkIGZyb20gb3JpZ2luYWwgYmFzZWxpbmUpCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBTdGFydGVyUGxhbm5lckNvbmZpZzoKICAgIHBsYW5uZXJfdHlwZTogc3RyID0gInN0YXJ0ZXJfcGQiCiAgICBzcGVlZF9tcHM6IGZsb2F0ID0gMC40NQogICAgbWluX3NwZWVkX21wczogZmxvYXQgPSAwLjEyCiAgICBtYXhfbGF0ZXJhbF9zcGVlZF9tcHM6IGZsb2F0ID0gMC4wOAogICAgbWF4X3lhd19yYXRlX3JhZHBzOiBmbG9hdCA9IDAuMjUKICAgIGtfaGVhZGluZzogZmxvYXQgPSAwLjU1CiAgICBrX2xhdGVyYWw6IGZsb2F0ID0gMC4wOAogICAgaGVhZGluZ19zbG93ZG93bjogZmxvYXQgPSAwLjQ1CiAgICBzdGFuZF9zZWNvbmRzOiBmbG9hdCA9IDEuMAoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGZyb21fZGljdChjbHMsIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldKSAtPiAiU3RhcnRlclBsYW5uZXJDb25maWciOgogICAgICAgIHZhbGlkID0gc2V0KGNscy5fX2RhdGFjbGFzc19maWVsZHNfXy5rZXlzKCkpCiAgICAgICAgcmV0dXJuIGNscygqKntrOiBwYXlsb2FkW2tdIGZvciBrIGluIHZhbGlkIGlmIGsgaW4gcGF5bG9hZH0pCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgbG9hZChjbHMsIHBhdGg6IFBhdGgpIC0+ICJTdGFydGVyUGxhbm5lckNvbmZpZyI6CiAgICAgICAgcmV0dXJuIGNscy5mcm9tX2RpY3QoanNvbi5sb2FkcyhQYXRoKHBhdGgpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkpCgogICAgZGVmIHRvX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInBsYW5uZXJfdHlwZSI6IHNlbGYucGxhbm5lcl90eXBlLAogICAgICAgICAgICAic3BlZWRfbXBzIjogc2VsZi5zcGVlZF9tcHMsCiAgICAgICAgICAgICJtaW5fc3BlZWRfbXBzIjogc2VsZi5taW5fc3BlZWRfbXBzLAogICAgICAgICAgICAibWF4X2xhdGVyYWxfc3BlZWRfbXBzIjogc2VsZi5tYXhfbGF0ZXJhbF9zcGVlZF9tcHMsCiAgICAgICAgICAgICJtYXhfeWF3X3JhdGVfcmFkcHMiOiBzZWxmLm1heF95YXdfcmF0ZV9yYWRwcywKICAgICAgICAgICAgImtfaGVhZGluZyI6IHNlbGYua19oZWFkaW5nLAogICAgICAgICAgICAia19sYXRlcmFsIjogc2VsZi5rX2xhdGVyYWwsCiAgICAgICAgICAgICJoZWFkaW5nX3Nsb3dkb3duIjogc2VsZi5oZWFkaW5nX3Nsb3dkb3duLAogICAgICAgICAgICAic3RhbmRfc2Vjb25kcyI6IHNlbGYuc3RhbmRfc2Vjb25kcywKICAgICAgICB9CgoKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKIyBNTFBUcmFja1BsYW5uZXIgIOKAkyBsZWFybmVkIGhpZ2gtbGV2ZWwgY29udHJvbGxlcgojIOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkAoKY2xhc3MgTUxQVHJhY2tQbGFubmVyOgogICAgIiIiVHdvLWhpZGRlbi1sYXllciBNTFAgdGhhdCBtYXBzIHRoZSA1LUQgdHJhY2sgb2JzZXJ2YXRpb24gdG8gW3Z4LCB2eSwgeWF3X3JhdGVdLgoKICAgIEFyY2hpdGVjdHVyZSAoZGVmYXVsdCk6ICA1IOKGkiAzMiDihpIgMTYg4oaSIDMgIHdpdGggdGFuaCBhY3RpdmF0aW9ucy4KCiAgICBUaGUgb3V0cHV0IGxheWVyIHVzZXMgdGFuaCBzbyBhbGwgdGhyZWUgb3V0cHV0cyBsaWUgaW4gKOKIkjEsIDEpLCB3aGljaCBhcmUKICAgIHRoZW4gYWZmaW5lbHkgbWFwcGVkIHRvIHRoZSBwaHlzaWNhbCBjb21tYW5kIHJhbmdlczoKCiAgICAgICAgdnggICAgICAgIOKIiCBbX1ZYX01JTiwgX1ZYX01BWF0gICAoYWx3YXlzIHBvc2l0aXZlIOKGkiByb2JvdCBtb3ZlcyBmb3J3YXJkKQogICAgICAgIHZ5ICAgICAgICDiiIggW+KIkl9WWV9MSU0sIF9WWV9MSU1dCiAgICAgICAgeWF3X3JhdGUgIOKIiCBb4oiSX1lBV19MSU0sIF9ZQVdfTElNXQoKICAgIFdlaWdodHMgYXJlIHN0b3JlZCBhcyBhIC5ucHogZmlsZS4gIFRoZSBKU09OIGNvbmZpZyBtdXN0IGNvbnRhaW46CgogICAgICAgIHsKICAgICAgICAgICJwbGFubmVyX3R5cGUiOiAibWxwIiwKICAgICAgICAgICJtbHBfd2VpZ2h0c19wYXRoIjogInBsYW5uZXJfd2VpZ2h0cy5ucHoiLCAgIC8vIHJlbGF0aXZlIHRvIGNvbmZpZyBkaXIKICAgICAgICAgICJtbHBfaGlkZGVuIjogWzMyLCAxNl0sICAgICAgICAgICAgICAgICAgICAgICAvLyBoaWRkZW4gbGF5ZXIgd2lkdGhzCiAgICAgICAgICAic3RhbmRfc2Vjb25kcyI6IDEuMAogICAgICAgIH0KICAgICIiIgoKICAgIF9PQlNfU0laRSA9IDUKICAgIF9DTURfU0laRSA9IDMKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICB3ZWlnaHRzOiBkaWN0W3N0ciwgbnAubmRhcnJheV0sCiAgICAgICAgaGlkZGVuX3NpemVzOiBsaXN0W2ludF0sCiAgICAgICAgc3RhbmRfc2Vjb25kczogZmxvYXQgPSAxLjAsCiAgICApIC0+IE5vbmU6CiAgICAgICAgc2VsZi53ZWlnaHRzID0ge2s6IHYuYXN0eXBlKG5wLmZsb2F0MzIpIGZvciBrLCB2IGluIHdlaWdodHMuaXRlbXMoKX0KICAgICAgICBzZWxmLmhpZGRlbl9zaXplcyA9IGxpc3QoaGlkZGVuX3NpemVzKQogICAgICAgIHNlbGYuc3RhbmRfc2Vjb25kcyA9IGZsb2F0KHN0YW5kX3NlY29uZHMpCiAgICAgICAgc2VsZi5fbl9sYXllcnMgPSBsZW4oaGlkZGVuX3NpemVzKSArIDEKCiAgICAjIOKUgOKUgCBDb25zdHJ1Y3Rpb24g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgbG9hZChjbHMsIHBhdGg6IFBhdGgpIC0+ICJNTFBUcmFja1BsYW5uZXIiOgogICAgICAgICIiIkxvYWQgZnJvbSBhIEpTT04gY29uZmlnIGZpbGUgKHdlaWdodHMgcGF0aCBpcyByZWxhdGl2ZSB0byBjb25maWcpLiIiIgogICAgICAgIGNmZyA9IGpzb24ubG9hZHMoUGF0aChwYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgd2VpZ2h0c19wYXRoID0gUGF0aChwYXRoKS5wYXJlbnQgLyBjZmdbIm1scF93ZWlnaHRzX3BhdGgiXQogICAgICAgIHdlaWdodHMgPSBkaWN0KG5wLmxvYWQoc3RyKHdlaWdodHNfcGF0aCkpKQogICAgICAgIHJldHVybiBjbHMoCiAgICAgICAgICAgIHdlaWdodHM9d2VpZ2h0cywKICAgICAgICAgICAgaGlkZGVuX3NpemVzPWNmZy5nZXQoIm1scF9oaWRkZW4iLCBbMzIsIDE2XSksCiAgICAgICAgICAgIHN0YW5kX3NlY29uZHM9ZmxvYXQoY2ZnLmdldCgic3RhbmRfc2Vjb25kcyIsIDEuMCkpLAogICAgICAgICkKCiAgICAjIOKUgOKUgCBGb3J3YXJkIHBhc3Mg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgZGVmIF9mb3J3YXJkKHNlbGYsIHg6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIiIiUHVyZSBOdW1QeSBmb3J3YXJkIHBhc3MuICB4IHNoYXBlOiAoNSwpLiAgUmV0dXJucyBzaGFwZSAoMywpLiIiIgogICAgICAgIGZvciBpIGluIHJhbmdlKHNlbGYuX25fbGF5ZXJzKToKICAgICAgICAgICAgeCA9IHggQCBzZWxmLndlaWdodHNbZiJXe2l9Il0gKyBzZWxmLndlaWdodHNbZiJie2l9Il0KICAgICAgICAgICAgaWYgaSA8IHNlbGYuX25fbGF5ZXJzIC0gMToKICAgICAgICAgICAgICAgIHggPSBucC50YW5oKHgpCiAgICAgICAgcmV0dXJuIG5wLnRhbmgoeCkgICMgZmluYWwgYWN0aXZhdGlvbiDihpIgKOKIkjEsIDEpCgogICAgIyDilIDilIAgUGxhbm5lciBBUEkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgZGVmIGNvbW1hbmQoc2VsZiwgb2JzOiBUcmFja0NvbnRyb2xsZXJPYnNlcnZhdGlvbiwgdDogZmxvYXQpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIiIiUmV0dXJuIFt2eCwgdnksIHlhd19yYXRlXSBjb21tYW5kIGdpdmVuIHRyYWNrIG9ic2VydmF0aW9uIGFuZCB0aW1lLiIiIgogICAgICAgIGlmIHQgPCBzZWxmLnN0YW5kX3NlY29uZHM6CiAgICAgICAgICAgIHJldHVybiBucC56ZXJvcygzLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIHJhdyA9IHNlbGYuX2ZvcndhcmQob2JzLmFzX2FycmF5KCkpCiAgICAgICAgIyBNYXAgdGFuaCBvdXRwdXRzIHRvIHBoeXNpY2FsIHJhbmdlcwogICAgICAgIHZ4ID0gKHJhd1swXSArIDEuMCkgKiAwLjUgKiAoX1ZYX01BWCAtIF9WWF9NSU4pICsgX1ZYX01JTgogICAgICAgIHZ5ID0gcmF3WzFdICogX1ZZX0xJTQogICAgICAgIHlhd19yYXRlID0gcmF3WzJdICogX1lBV19MSU0KICAgICAgICByZXR1cm4gbnAuYXJyYXkoW3Z4LCB2eSwgeWF3X3JhdGVdLCBkdHlwZT1ucC5mbG9hdDMyKQoKICAgICMg4pSA4pSAIFdlaWdodCBoZWxwZXJzIHVzZWQgYnkgdGhlIHRyYWluaW5nIHNjcmlwdCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgbWFrZV93ZWlnaHRzKGhpZGRlbl9zaXplczogbGlzdFtpbnRdLCBzZWVkOiBpbnQgPSA0MikgLT4gZGljdFtzdHIsIG5wLm5kYXJyYXldOgogICAgICAgICIiIkhlLWluaXRpYWxpc2VkIHdlaWdodHM7IG91dHB1dCBiaWFzIHNldCB0byBnaXZlIHZ44omIMC4zNSBhdCBzdGFydC4iIiIKICAgICAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkKICAgICAgICBzaXplcyA9IFtNTFBUcmFja1BsYW5uZXIuX09CU19TSVpFXSArIGhpZGRlbl9zaXplcyArIFtNTFBUcmFja1BsYW5uZXIuX0NNRF9TSVpFXQogICAgICAgIHdlaWdodHM6IGRpY3Rbc3RyLCBucC5uZGFycmF5XSA9IHt9CiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKHNpemVzKSAtIDEpOgogICAgICAgICAgICBmYW5faW4gPSBzaXplc1tpXQogICAgICAgICAgICB3ZWlnaHRzW2YiV3tpfSJdID0gKAogICAgICAgICAgICAgICAgcm5nLnN0YW5kYXJkX25vcm1hbCgoZmFuX2luLCBzaXplc1tpICsgMV0pKSAqIG5wLnNxcnQoMi4wIC8gZmFuX2luKQogICAgICAgICAgICApLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgICAgICB3ZWlnaHRzW2YiYntpfSJdID0gbnAuemVyb3Moc2l6ZXNbaSArIDFdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICMgSW5pdCBvdXRwdXQgYmlhcyBzbyB2eCDiiYggMC4zNSBtL3MgKG1pZC1yYW5nZSksIHZ5PXlhdz0wCiAgICAgICAgIyB0YW5oKGIpID0gKDAuMzUgLSBWWF9NSU4pIC8gKFZYX01BWCAtIFZYX01JTikgKiAyIC0gMQogICAgICAgIHZ4X2luaXRfbm9ybSA9ICgwLjM1IC0gX1ZYX01JTikgLyAoX1ZYX01BWCAtIF9WWF9NSU4pICogMi4wIC0gMS4wCiAgICAgICAgbGFzdCA9IGYiYntsZW4oc2l6ZXMpLTJ9IgogICAgICAgIHdlaWdodHNbbGFzdF1bMF0gPSBmbG9hdChucC5hcmN0YW5oKG5wLmNsaXAodnhfaW5pdF9ub3JtLCAtMC45OSwgMC45OSkpKQogICAgICAgIHJldHVybiB3ZWlnaHRzCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIHBhY2sod2VpZ2h0czogZGljdFtzdHIsIG5wLm5kYXJyYXldKSAtPiBucC5uZGFycmF5OgogICAgICAgICIiIkZsYXR0ZW4gd2VpZ2h0IGRpY3QgdG8gYSAxLUQgZmxvYXQ2NCBhcnJheSBmb3Igb3B0aW1pc2Vycy4iIiIKICAgICAgICByZXR1cm4gbnAuY29uY2F0ZW5hdGUoCiAgICAgICAgICAgIFt3ZWlnaHRzW2tdLnJhdmVsKCkuYXN0eXBlKG5wLmZsb2F0NjQpIGZvciBrIGluIHNvcnRlZCh3ZWlnaHRzKV0KICAgICAgICApCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIHVucGFjayh0aGV0YTogbnAubmRhcnJheSwgaGlkZGVuX3NpemVzOiBsaXN0W2ludF0pIC0+IGRpY3Rbc3RyLCBucC5uZGFycmF5XToKICAgICAgICAiIiJSZXN0b3JlIHdlaWdodCBkaWN0IGZyb20gZmxhdCAxLUQgYXJyYXkuIiIiCiAgICAgICAgc2l6ZXMgPSBbTUxQVHJhY2tQbGFubmVyLl9PQlNfU0laRV0gKyBoaWRkZW5fc2l6ZXMgKyBbTUxQVHJhY2tQbGFubmVyLl9DTURfU0laRV0KICAgICAgICB3ZWlnaHRzOiBkaWN0W3N0ciwgbnAubmRhcnJheV0gPSB7fQogICAgICAgIG9mZnNldCA9IDAKICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oc2l6ZXMpIC0gMSk6CiAgICAgICAgICAgIG5fVyA9IHNpemVzW2ldICogc2l6ZXNbaSArIDFdCiAgICAgICAgICAgIG5fYiA9IHNpemVzW2kgKyAxXQogICAgICAgICAgICB3ZWlnaHRzW2YiV3tpfSJdID0gdGhldGFbb2Zmc2V0OiBvZmZzZXQgKyBuX1ddLnJlc2hhcGUoCiAgICAgICAgICAgICAgICBzaXplc1tpXSwgc2l6ZXNbaSArIDFdCiAgICAgICAgICAgICkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgICAgIG9mZnNldCArPSBuX1cKICAgICAgICAgICAgd2VpZ2h0c1tmImJ7aX0iXSA9IHRoZXRhW29mZnNldDogb2Zmc2V0ICsgbl9iXS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICAgICAgb2Zmc2V0ICs9IG5fYgogICAgICAgIHJldHVybiB3ZWlnaHRzCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIHBhcmFtX2NvdW50KGhpZGRlbl9zaXplczogbGlzdFtpbnRdKSAtPiBpbnQ6CiAgICAgICAgc2l6ZXMgPSBbTUxQVHJhY2tQbGFubmVyLl9PQlNfU0laRV0gKyBoaWRkZW5fc2l6ZXMgKyBbTUxQVHJhY2tQbGFubmVyLl9DTURfU0laRV0KICAgICAgICByZXR1cm4gc3VtKAogICAgICAgICAgICBzaXplc1tpXSAqIHNpemVzW2kgKyAxXSArIHNpemVzW2kgKyAxXSBmb3IgaSBpbiByYW5nZShsZW4oc2l6ZXMpIC0gMSkKICAgICAgICApCgoKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKIyBTdGFydGVyVHJhY2tQbGFubmVyICDigJMgZXZhbHVhdG9yIGVudHJ5IHBvaW50IChkaXNwYXRjaGVzIG9uIHBsYW5uZXJfdHlwZSkKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKCmNsYXNzIFN0YXJ0ZXJUcmFja1BsYW5uZXI6CiAgICAiIiJDb25zZXJ2YXRpdmUgUEQgYmFzZWxpbmU7IGFsc28gZGlzcGF0Y2hlcyBgYGxvYWQoKWBgIHRvIE1MUFRyYWNrUGxhbm5lci4KCiAgICBTdHVkZW50cyBzaG91bGQgaW1wcm92ZSB0aGlzIGNsYXNzLCByZXBsYWNlIGl0IHdpdGggYW4gTUxQLCBvciB0cmFpbiBhCiAgICBoaWdoZXItbGV2ZWwgcG9saWN5IHRoYXQgcHJvZHVjZXMgdGhlIHNhbWUgW3Z4LCB2eSwgeWF3X3JhdGVdIGNvbW1hbmQuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnOiBTdGFydGVyUGxhbm5lckNvbmZpZykgLT4gTm9uZToKICAgICAgICBpZiBjb25maWcucGxhbm5lcl90eXBlICE9ICJzdGFydGVyX3BkIjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuc3VwcG9ydGVkIHBsYW5uZXJfdHlwZToge2NvbmZpZy5wbGFubmVyX3R5cGUhcn0iKQogICAgICAgIHNlbGYuY29uZmlnID0gY29uZmlnCiAgICAgICAgc2VsZi50cmFjazogU3RhbmRhcmRPdmFsVHJhY2sgPSBvZmZpY2lhbF90cmFjaygpCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgbG9hZChjbHMsIHBhdGg6IFBhdGgpIC0+IFVuaW9uWyJTdGFydGVyVHJhY2tQbGFubmVyIiwgTUxQVHJhY2tQbGFubmVyXToKICAgICAgICAiIiJMb2FkIHBsYW5uZXIgZnJvbSBKU09OIGNvbmZpZy4gIERpc3BhdGNoZXMgdG8gTUxQVHJhY2tQbGFubmVyIGZvciAnbWxwJy4iIiIKICAgICAgICByYXcgPSBqc29uLmxvYWRzKFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGlmIHJhdy5nZXQoInBsYW5uZXJfdHlwZSIpID09ICJtbHAiOgogICAgICAgICAgICByZXR1cm4gTUxQVHJhY2tQbGFubmVyLmxvYWQocGF0aCkKICAgICAgICByZXR1cm4gY2xzKFN0YXJ0ZXJQbGFubmVyQ29uZmlnLmxvYWQocGF0aCkpCgogICAgZGVmIGNvbW1hbmQoc2VsZiwgb2JzOiBUcmFja0NvbnRyb2xsZXJPYnNlcnZhdGlvbiwgdDogZmxvYXQpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgaWYgdCA8IHNlbGYuY29uZmlnLnN0YW5kX3NlY29uZHM6CiAgICAgICAgICAgIHJldHVybiBucC56ZXJvcygzLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIHJldHVybiBzZWxmLmNvbW1hbmRfZnJvbV9vYnNlcnZhdGlvbihvYnMpCgogICAgZGVmIGNvbW1hbmRfZnJvbV9vYnNlcnZhdGlvbihzZWxmLCBvYnM6IFRyYWNrQ29udHJvbGxlck9ic2VydmF0aW9uKSAtPiBucC5uZGFycmF5OgogICAgICAgIGxhdGVyYWxfZXJyb3IgPSBmbG9hdChvYnMubGF0ZXJhbF9lcnJvcl9ub3JtKSAqIGZsb2F0KHNlbGYudHJhY2suaGFsZl93aWR0aF9tKQogICAgICAgIGxhdGVyYWxfYmlhcyA9IG1hdGguYXRhbjIoCiAgICAgICAgICAgIGZsb2F0KHNlbGYuY29uZmlnLmtfbGF0ZXJhbCkgKiBsYXRlcmFsX2Vycm9yLAogICAgICAgICAgICBtYXgoZmxvYXQoc2VsZi5jb25maWcuc3BlZWRfbXBzKSwgMWUtMyksCiAgICAgICAgKQogICAgICAgIGhlYWRpbmdfZXJyb3IgPSB3cmFwX2FuZ2xlKGZsb2F0KG9icy5oZWFkaW5nX2Vycm9yX3JhZCkgLSBsYXRlcmFsX2JpYXMpCgogICAgICAgIHNwZWVkX3NjYWxlID0gKAogICAgICAgICAgICAxLjAKICAgICAgICAgICAgLSBmbG9hdChzZWxmLmNvbmZpZy5oZWFkaW5nX3Nsb3dkb3duKQogICAgICAgICAgICAqIG1pbihhYnMoaGVhZGluZ19lcnJvciksIG1hdGgucGkpCiAgICAgICAgICAgIC8gbWF0aC5waQogICAgICAgICkKICAgICAgICB2eCA9IG5wLmNsaXAoCiAgICAgICAgICAgIGZsb2F0KHNlbGYuY29uZmlnLnNwZWVkX21wcykgKiBzcGVlZF9zY2FsZSwKICAgICAgICAgICAgZmxvYXQoc2VsZi5jb25maWcubWluX3NwZWVkX21wcyksCiAgICAgICAgICAgIGZsb2F0KHNlbGYuY29uZmlnLnNwZWVkX21wcyksCiAgICAgICAgKQogICAgICAgIHZ5ID0gbnAuY2xpcCgKICAgICAgICAgICAgLWZsb2F0KHNlbGYuY29uZmlnLmtfbGF0ZXJhbCkgKiBsYXRlcmFsX2Vycm9yLAogICAgICAgICAgICAtZmxvYXQoc2VsZi5jb25maWcubWF4X2xhdGVyYWxfc3BlZWRfbXBzKSwKICAgICAgICAgICAgZmxvYXQoc2VsZi5jb25maWcubWF4X2xhdGVyYWxfc3BlZWRfbXBzKSwKICAgICAgICApCiAgICAgICAgY3VydmF0dXJlID0gZmxvYXQob2JzLmN1cnZhdHVyZV9ub3JtKSAvIG1heCgKICAgICAgICAgICAgZmxvYXQoc2VsZi50cmFjay50dXJuX3JhZGl1c19tKSwgMWUtNgogICAgICAgICkKICAgICAgICB5YXdfcmF0ZSA9IG5wLmNsaXAoCiAgICAgICAgICAgIGN1cnZhdHVyZSAqIHZ4ICsgZmxvYXQoc2VsZi5jb25maWcua19oZWFkaW5nKSAqIGhlYWRpbmdfZXJyb3IsCiAgICAgICAgICAgIC1mbG9hdChzZWxmLmNvbmZpZy5tYXhfeWF3X3JhdGVfcmFkcHMpLAogICAgICAgICAgICBmbG9hdChzZWxmLmNvbmZpZy5tYXhfeWF3X3JhdGVfcmFkcHMpLAogICAgICAgICkKICAgICAgICByZXR1cm4gbnAuYXNhcnJheShbdngsIHZ5LCB5YXdfcmF0ZV0sIGR0eXBlPW5wLmZsb2F0MzIpCg=='
(COURSE_REPO_DIR / 'track_bonus' / 'planner.py').write_text(
    _b64.b64decode(_planner_b64).decode(), encoding='utf-8')
print('Patched track_bonus/planner.py with MLPTrackPlanner.')

_train_b64 = 'IiIiQ01BLUVTIHRyYWluaW5nIHNjcmlwdCBmb3IgdGhlIGxlYXJuZWQgTUxQIGhpZ2gtbGV2ZWwgdHJhY2sgcGxhbm5lci4KClVzYWdlIChmcm9tIHRoZSBjb3Vyc2UgcmVwbyByb290IGluIENvbGFiKToKCiAgICBweXRob24gdHJhaW5faGlnaGxldmVsX21scC5weSBcXAogICAgICAgIC0tY2hlY2twb2ludC1kaXIgYXJ0aWZhY3RzL2xvd19sZXZlbF90cmFpbi9iZXN0X2NoZWNrcG9pbnQgXFwKICAgICAgICAtLWNvbmZpZyBjb25maWdzL2NvbGFiX3J1bnRpbWVfY29uZmlnLmpzb24gXFwKICAgICAgICAtLW91dHB1dC1kaXIgYXJ0aWZhY3RzL2hpZ2hsZXZlbF9tbHAgXFwKICAgICAgICAtLWl0ZXJhdGlvbnMgNDAgXFwKICAgICAgICAtLXBvcHVsYXRpb24gMTAgXFwKICAgICAgICAtLWV2YWwtc2Vjb25kcyAxNQoKVGhlIHNjcmlwdDoKICAxLiBJbml0aWFsaXNlcyBNTFAgd2VpZ2h0cyAoNeKGkjMy4oaSMTbihpIzLCB0YW5oKS4KICAyLiBSdW5zIENNQS1FUzogZWFjaCBjYW5kaWRhdGUgc2F2ZXMgYSB0ZW1wIGNvbmZpZyt3ZWlnaHRzLCBjYWxscwogICAgIHJ1bl90cmFja19ib251cy5weSwgYW5kIHJlYWRzIGNvbXBvc2l0ZV9zY29yZSBmcm9tIHJlc3VsdHMuanNvbi4KICAzLiBTYXZlcyBiZXN0IHdlaWdodHMgdG8gPG91dHB1dC1kaXI+L3BsYW5uZXJfd2VpZ2h0cy5ucHogYW5kIHRoZQogICAgIG1hdGNoaW5nIGNvbmZpZyB0byA8b3V0cHV0LWRpcj4vcGxhbm5lcl9jb25maWcuanNvbi4KICA0LiBXcml0ZXMgYSBzZWFyY2ggaGlzdG9yeSB0byA8b3V0cHV0LWRpcj4vc2VhcmNoX2hpc3RvcnkuanNvbi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKCiMg4pSA4pSAIG1ha2Ugc3VyZSB0aGUgY291cnNlIHJlcG8gaXMgaW1wb3J0YWJsZSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5wYXJlbnQpKQpmcm9tIHRyYWNrX2JvbnVzLnBsYW5uZXIgaW1wb3J0IE1MUFRyYWNrUGxhbm5lcgoKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKIyBNaW5pbWFsIENNQS1FUyAgKG5vIGV4dGVybmFsIGRlcGVuZGVuY3kpCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCgpjbGFzcyBfQ01BZXM6CiAgICAiIiJTaW1wbGlmaWVkICjOvC/OvF93LCDOuyktQ01BLUVTIHdpdGhvdXQgY292YXJpYW5jZSBtYXRyaXggdXBkYXRlLgoKICAgIFVzZXMgYSBkaWFnb25hbCBhZGFwdGF0aW9uIHNvIGl0IHNjYWxlcyB0byB+MTUwLTMwMCBwYXJhbWV0ZXJzIHdpdGhvdXQKICAgIHRoZSBPKG7CsikgY292YXJpYW5jZSBjb3N0LiAgR29vZCBlbm91Z2ggZm9yIGJsYWNrLWJveCBwbGFubmVyIHR1bmluZy4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHgwOiBucC5uZGFycmF5LAogICAgICAgIHNpZ21hMDogZmxvYXQgPSAwLjMsCiAgICAgICAgcG9wc2l6ZTogaW50ID0gMTAsCiAgICAgICAgc2VlZDogaW50ID0gMCwKICAgICkgLT4gTm9uZToKICAgICAgICBzZWxmLnJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgICAgIHNlbGYubiA9IGxlbih4MCkKICAgICAgICBzZWxmLm1lYW4gPSB4MC5jb3B5KCkuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICAgICAgc2VsZi5zaWdtYSA9IGZsb2F0KHNpZ21hMCkKICAgICAgICBzZWxmLmxhbSA9IHBvcHNpemUKICAgICAgICBzZWxmLm11ID0gbWF4KHBvcHNpemUgLy8gMiwgMikKCiAgICAgICAgIyBSZWNvbWJpbmF0aW9uIHdlaWdodHMKICAgICAgICByYXdfdyA9IG5wLmxvZyhzZWxmLm11ICsgMC41KSAtIG5wLmxvZyhucC5hcmFuZ2UoMSwgc2VsZi5tdSArIDEpKQogICAgICAgIHNlbGYudyA9IHJhd193IC8gcmF3X3cuc3VtKCkKICAgICAgICBzZWxmLm11ZWZmID0gMS4wIC8gZmxvYXQobnAuc3VtKHNlbGYudyAqKiAyKSkKCiAgICAgICAgIyBTdGVwLXNpemUgYWRhcHRhdGlvbiBjb2VmZmljaWVudHMgKENTQSkKICAgICAgICBzZWxmLmNzID0gKHNlbGYubXVlZmYgKyAyLjApIC8gKHNlbGYubiArIHNlbGYubXVlZmYgKyA1LjApCiAgICAgICAgc2VsZi5kcyA9IDEuMCArIDIuMCAqIG1heCgwLjAsIG5wLnNxcnQoKHNlbGYubXVlZmYgLSAxLjApIC8gKHNlbGYubiArIDEuMCkpIC0gMS4wKSArIHNlbGYuY3MKICAgICAgICBzZWxmLmNoaU4gPSBmbG9hdChucC5zcXJ0KHNlbGYubikgKiAoMS4wIC0gMS4wIC8gKDQuMCAqIHNlbGYubikgKyAxLjAgLyAoMjEuMCAqIHNlbGYubiAqKiAyKSkpCiAgICAgICAgc2VsZi5wcyA9IG5wLnplcm9zKHNlbGYubikKCiAgICAgICAgIyBEaWFnb25hbCB2YXJpYW5jZSAgKHJlcGxhY2VzIGZ1bGwgY292YXJpYW5jZSkKICAgICAgICBzZWxmLnZhciA9IG5wLm9uZXMoc2VsZi5uKQoKICAgICMg4pSA4pSAIHB1YmxpYyBBUEkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgZGVmIGFzayhzZWxmKSAtPiBucC5uZGFycmF5OgogICAgICAgICIiIlJldHVybiAobGFtYmRhLCBuKSBhcnJheSBvZiBjYW5kaWRhdGUgcGFyYW1ldGVyIHZlY3RvcnMuIiIiCiAgICAgICAgeiA9IHNlbGYucm5nLnN0YW5kYXJkX25vcm1hbCgoc2VsZi5sYW0sIHNlbGYubikpCiAgICAgICAgcmV0dXJuIHNlbGYubWVhbiArIHNlbGYuc2lnbWEgKiBucC5zcXJ0KHNlbGYudmFyKSAqIHoKCiAgICBkZWYgdGVsbChzZWxmLCB4czogbnAubmRhcnJheSwgc2NvcmVzOiBucC5uZGFycmF5KSAtPiBOb25lOgogICAgICAgICIiIlVwZGF0ZSBkaXN0cmlidXRpb24gZ2l2ZW4gY2FuZGlkYXRlcyB4cyBhbmQgdGhlaXIgZml0bmVzc2VzIChoaWdoZXI9YmV0dGVyKS4iIiIKICAgICAgICBvcmRlciA9IG5wLmFyZ3NvcnQoLXNjb3JlcykgICAgICAgICAgIyBkZXNjZW5kaW5nCiAgICAgICAgZWxpdGUgPSB4c1tvcmRlcls6IHNlbGYubXVdXSAgICAgICAgICMgdG9wLc68IGNhbmRpZGF0ZXMKCiAgICAgICAgb2xkX21lYW4gPSBzZWxmLm1lYW4uY29weSgpCiAgICAgICAgc2VsZi5tZWFuID0gKHNlbGYud1s6LCBOb25lXSAqIGVsaXRlKS5zdW0oYXhpcz0wKQoKICAgICAgICAjIFN0ZXAgaW4gbm9ybWFsaXNlZCBzcGFjZQogICAgICAgIHN0ZXAgPSAoc2VsZi5tZWFuIC0gb2xkX21lYW4pIC8gKHNlbGYuc2lnbWEgKiBucC5zcXJ0KHNlbGYudmFyKSArIDFlLTEyKQogICAgICAgICMgQ1NBIHBhdGggYW5kIHNpZ21hIHVwZGF0ZQogICAgICAgIHNlbGYucHMgPSAoMS4wIC0gc2VsZi5jcykgKiBzZWxmLnBzICsgbnAuc3FydCgKICAgICAgICAgICAgc2VsZi5jcyAqICgyLjAgLSBzZWxmLmNzKSAqIHNlbGYubXVlZmYKICAgICAgICApICogc3RlcAogICAgICAgIHNlbGYuc2lnbWEgKj0gZmxvYXQobnAuZXhwKChzZWxmLmNzIC8gc2VsZi5kcykgKiAobnAubGluYWxnLm5vcm0oc2VsZi5wcykgLyBzZWxmLmNoaU4gLSAxLjApKSkKICAgICAgICBzZWxmLnNpZ21hID0gZmxvYXQobnAuY2xpcChzZWxmLnNpZ21hLCAxZS04LCAyLjApKQoKICAgICAgICAjIERpYWdvbmFsIHZhcmlhbmNlIGFkYXB0YXRpb24gKGN1bXVsYXRpdmUgcmFuay1vbmUgdXBkYXRlKQogICAgICAgIHlzID0gKGVsaXRlIC0gb2xkX21lYW4pIC8gKHNlbGYuc2lnbWEgKiBucC5zcXJ0KHNlbGYudmFyKSArIDFlLTEyKQogICAgICAgIHNlbGYudmFyID0gMC45ICogc2VsZi52YXIgKyAwLjEgKiBmbG9hdChucC5zdW0oc2VsZi53KSkgKiAoc2VsZi53WzosIE5vbmVdICogeXMgKiogMikuc3VtKGF4aXM9MCkKICAgICAgICBzZWxmLnZhciA9IG5wLmNsaXAoc2VsZi52YXIsIDFlLTEwLCBOb25lKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGJlc3Qoc2VsZikgLT4gbnAubmRhcnJheToKICAgICAgICByZXR1cm4gc2VsZi5tZWFuLmNvcHkoKQoKCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCiMgRXZhbHVhdGlvbiBoZWxwZXIKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKCmRlZiBfZXZhbHVhdGUoCiAgICB0aGV0YTogbnAubmRhcnJheSwKICAgIGhpZGRlbl9zaXplczogbGlzdFtpbnRdLAogICAgZXZhbF9kaXI6IFBhdGgsCiAgICBjaGVja3BvaW50X2RpcjogUGF0aCwKICAgIGNvbmZpZ19wYXRoOiBQYXRoLAogICAgcGxhbm5lcl9jb25maWdfcGF0aDogUGF0aCwKICAgIGV2YWxfc2Vjb25kczogZmxvYXQsCiAgICBzZWVkOiBpbnQsCiAgICBlbnRyeV9uYW1lOiBzdHIgPSAibWxwX2NhbmQiLAopIC0+IGZsb2F0OgogICAgIiIiU2F2ZSB3ZWlnaHRzLCBydW4gcnVuX3RyYWNrX2JvbnVzLnB5LCByZXR1cm4gY29tcG9zaXRlX3Njb3JlLiIiIgogICAgZXZhbF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgICMgV3JpdGUgd2VpZ2h0cwogICAgd2VpZ2h0cyA9IE1MUFRyYWNrUGxhbm5lci51bnBhY2sodGhldGEsIGhpZGRlbl9zaXplcykKICAgIHdlaWdodHNfcGF0aCA9IGV2YWxfZGlyIC8gInBsYW5uZXJfd2VpZ2h0cy5ucHoiCiAgICBucC5zYXZleihzdHIod2VpZ2h0c19wYXRoKSwgKip3ZWlnaHRzKQoKICAgICMgV3JpdGUgY29uZmlnIHBvaW50aW5nIGF0IHRoZXNlIHdlaWdodHMKICAgIGNmZyA9IGpzb24ubG9hZHMocGxhbm5lcl9jb25maWdfcGF0aC5yZWFkX3RleHQoKSkKICAgIGNmZ1sibWxwX3dlaWdodHNfcGF0aCJdID0gInBsYW5uZXJfd2VpZ2h0cy5ucHoiCiAgICB0bXBfY2ZnID0gZXZhbF9kaXIgLyAicGxhbm5lcl9jb25maWcuanNvbiIKICAgIHRtcF9jZmcud3JpdGVfdGV4dChqc29uLmR1bXBzKGNmZywgaW5kZW50PTIpKQoKICAgICMgUnVuIGV2YWx1YXRpb24gKHNob3J0LCBubyB2aWRlbykKICAgIGNtZCA9IFsKICAgICAgICBzeXMuZXhlY3V0YWJsZSwgInJ1bl90cmFja19ib251cy5weSIsCiAgICAgICAgIi0tY2hlY2twb2ludC1kaXIiLCBzdHIoY2hlY2twb2ludF9kaXIpLAogICAgICAgICItLXBsYW5uZXItY29uZmlnIiwgc3RyKHRtcF9jZmcpLAogICAgICAgICItLWNvbmZpZyIsIHN0cihjb25maWdfcGF0aCksCiAgICAgICAgIi0tb3V0cHV0LWRpciIsIHN0cihldmFsX2RpciksCiAgICAgICAgIi0tZW50cnktbmFtZSIsIGVudHJ5X25hbWUsCiAgICAgICAgIi0tZHVyYXRpb24tc2Vjb25kcyIsIHN0cihldmFsX3NlY29uZHMpLAogICAgICAgICItLW5vLXJlbmRlciIsCiAgICAgICAgIi0tc2VlZCIsIHN0cihzZWVkKSwKICAgIF0KICAgIHRyeToKICAgICAgICBzdWJwcm9jZXNzLnJ1bihjbWQsIGNoZWNrPVRydWUsIHRpbWVvdXQ9ZXZhbF9zZWNvbmRzICogMTAgKyA2MCwKICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3VicHJvY2Vzcy5ERVZOVUxMLCBzdGRlcnI9c3VicHJvY2Vzcy5ERVZOVUxMKQogICAgZXhjZXB0IChzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvciwgc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZCkgYXMgZXhjOgogICAgICAgIHByaW50KGYiICAgIFt3YXJuXSBldmFsIGZhaWxlZDoge2V4Y30iKQogICAgICAgIHJldHVybiAwLjAKCiAgICByZXN1bHRzX2ZpbGUgPSBldmFsX2RpciAvICJyZXN1bHRzLmpzb24iCiAgICBpZiBub3QgcmVzdWx0c19maWxlLmV4aXN0cygpOgogICAgICAgIHJldHVybiAwLjAKICAgIHRyeToKICAgICAgICBkYXRhID0ganNvbi5sb2FkcyhyZXN1bHRzX2ZpbGUucmVhZF90ZXh0KCkpCiAgICAgICAgcmV0dXJuIGZsb2F0KGRhdGFbInNjb3JlcyJdWyJjb21wb3NpdGVfc2NvcmUiXSkKICAgIGV4Y2VwdCAoS2V5RXJyb3IsIGpzb24uSlNPTkRlY29kZUVycm9yKSBhcyBleGM6CiAgICAgICAgcHJpbnQoZiIgICAgW3dhcm5dIGNvdWxkIG5vdCBwYXJzZSByZXN1bHRzOiB7ZXhjfSIpCiAgICAgICAgcmV0dXJuIDAuMAoKCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCiMgTWFpbgojIOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkAoKZGVmIF9wYXJzZV9hcmdzKCkgLT4gYXJncGFyc2UuTmFtZXNwYWNlOgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJDTUEtRVMgdHJhaW5pbmcgZm9yIE1MUCB0cmFjayBwbGFubmVyIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWNoZWNrcG9pbnQtZGlyIiwgcmVxdWlyZWQ9VHJ1ZSwgdHlwZT1QYXRoKQogICAgcC5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD0iY29uZmlncy9jb2xhYl9ydW50aW1lX2NvbmZpZy5qc29uIiwgdHlwZT1QYXRoKQogICAgcC5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHJlcXVpcmVkPVRydWUsIHR5cGU9UGF0aCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLW1scC1oaWRkZW4iLCBkZWZhdWx0PSIzMiwxNiIsIGhlbHA9IkhpZGRlbiBsYXllciBzaXplcywgZS5nLiAnMzIsMTYnIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWl0ZXJhdGlvbnMiLCB0eXBlPWludCwgZGVmYXVsdD00MCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXBvcHVsYXRpb24iLCB0eXBlPWludCwgZGVmYXVsdD0xMCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXNpZ21hMCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4zKQogICAgcC5hZGRfYXJndW1lbnQoIi0tZXZhbC1zZWNvbmRzIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xNS4wKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTQyKQogICAgcmV0dXJuIHAucGFyc2VfYXJncygpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXJncyA9IF9wYXJzZV9hcmdzKCkKICAgIGhpZGRlbl9zaXplcyA9IFtpbnQoaCkgZm9yIGggaW4gYXJncy5tbHBfaGlkZGVuLnNwbGl0KCIsIildCiAgICBvdXRwdXRfZGlyOiBQYXRoID0gYXJncy5vdXRwdXRfZGlyCiAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBuX3BhcmFtcyA9IE1MUFRyYWNrUGxhbm5lci5wYXJhbV9jb3VudChoaWRkZW5fc2l6ZXMpCiAgICBwcmludChmIk1MUCBhcmNoaXRlY3R1cmU6IDXihpJ7J+KGkicuam9pbihzdHIoaCkgZm9yIGggaW4gaGlkZGVuX3NpemVzKX3ihpIzICAoe25fcGFyYW1zfSBwYXJhbWV0ZXJzKSIpCiAgICBwcmludChmIkNNQS1FUzoge2FyZ3MuaXRlcmF0aW9uc30gZ2VuZXJhdGlvbnMgw5cge2FyZ3MucG9wdWxhdGlvbn0gY2FuZGlkYXRlcyIpCiAgICBwcmludChmIkV2YWw6IHthcmdzLmV2YWxfc2Vjb25kc31zIHBlciBjYW5kaWRhdGUiKQogICAgcHJpbnQoZiJPdXRwdXQgZGlyOiB7b3V0cHV0X2Rpcn0iKQoKICAgICMg4pSA4pSAIEluaXRpYWwgd2VpZ2h0cyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIGluaXRfd2VpZ2h0cyA9IE1MUFRyYWNrUGxhbm5lci5tYWtlX3dlaWdodHMoaGlkZGVuX3NpemVzLCBzZWVkPWFyZ3Muc2VlZCkKICAgIHRoZXRhMCA9IE1MUFRyYWNrUGxhbm5lci5wYWNrKGluaXRfd2VpZ2h0cykKCiAgICAjIOKUgOKUgCBDcmVhdGUgYmFzZSBwbGFubmVyIGNvbmZpZyAobWxwIHR5cGUsIHdlaWdodHMgcmVsYXRpdmUgcGF0aCkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBiYXNlX3BsYW5uZXJfY2ZnID0gewogICAgICAgICJwbGFubmVyX3R5cGUiOiAibWxwIiwKICAgICAgICAibWxwX3dlaWdodHNfcGF0aCI6ICJwbGFubmVyX3dlaWdodHMubnB6IiwKICAgICAgICAibWxwX2hpZGRlbiI6IGhpZGRlbl9zaXplcywKICAgICAgICAic3RhbmRfc2Vjb25kcyI6IDEuMCwKICAgIH0KICAgIGJhc2VfY2ZnX3BhdGggPSBvdXRwdXRfZGlyIC8gIl9iYXNlX3BsYW5uZXJfY29uZmlnLmpzb24iCiAgICBiYXNlX2NmZ19wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhiYXNlX3BsYW5uZXJfY2ZnLCBpbmRlbnQ9MikpCgogICAgIyDilIDilIAgQ01BLUVTIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgZXMgPSBfQ01BZXModGhldGEwLCBzaWdtYTA9YXJncy5zaWdtYTAsIHBvcHNpemU9YXJncy5wb3B1bGF0aW9uLCBzZWVkPWFyZ3Muc2VlZCkKCiAgICBoaXN0b3J5OiBsaXN0W2RpY3RdID0gW10KICAgIGJlc3Rfc2NvcmUgPSAtMS4wCiAgICBiZXN0X3RoZXRhID0gdGhldGEwLmNvcHkoKQoKICAgIHRtcF9yb290ID0gb3V0cHV0X2RpciAvICJfY2FuZGlkYXRlcyIKCiAgICBmb3IgZ2VuIGluIHJhbmdlKGFyZ3MuaXRlcmF0aW9ucyk6CiAgICAgICAgdF9nZW5fc3RhcnQgPSB0aW1lLnRpbWUoKQogICAgICAgIHByaW50KGYiXG7ilIDilIAgR2VuZXJhdGlvbiB7Z2VuICsgMX0ve2FyZ3MuaXRlcmF0aW9uc30gICjPgz17ZXMuc2lnbWE6LjRmfSkg4pSA4pSAIikKCiAgICAgICAgY2FuZGlkYXRlcyA9IGVzLmFzaygpCiAgICAgICAgc2NvcmVzID0gbnAuemVyb3MoYXJncy5wb3B1bGF0aW9uKQoKICAgICAgICBmb3IgaWR4LCB0aGV0YSBpbiBlbnVtZXJhdGUoY2FuZGlkYXRlcyk6CiAgICAgICAgICAgIGNhbmRfZGlyID0gdG1wX3Jvb3QgLyBmImd7Z2VuOjAzZH1fY3tpZHg6MDJkfSIKICAgICAgICAgICAgc2NvcmUgPSBfZXZhbHVhdGUoCiAgICAgICAgICAgICAgICB0aGV0YT10aGV0YSwKICAgICAgICAgICAgICAgIGhpZGRlbl9zaXplcz1oaWRkZW5fc2l6ZXMsCiAgICAgICAgICAgICAgICBldmFsX2Rpcj1jYW5kX2RpciwKICAgICAgICAgICAgICAgIGNoZWNrcG9pbnRfZGlyPWFyZ3MuY2hlY2twb2ludF9kaXIsCiAgICAgICAgICAgICAgICBjb25maWdfcGF0aD1hcmdzLmNvbmZpZywKICAgICAgICAgICAgICAgIHBsYW5uZXJfY29uZmlnX3BhdGg9YmFzZV9jZmdfcGF0aCwKICAgICAgICAgICAgICAgIGV2YWxfc2Vjb25kcz1hcmdzLmV2YWxfc2Vjb25kcywKICAgICAgICAgICAgICAgIHNlZWQ9YXJncy5zZWVkICsgZ2VuICogMTAwICsgaWR4LAogICAgICAgICAgICAgICAgZW50cnlfbmFtZT1mImd7Z2VufWN7aWR4fSIsCiAgICAgICAgICAgICkKICAgICAgICAgICAgc2NvcmVzW2lkeF0gPSBzY29yZQogICAgICAgICAgICBpbmRpY2F0b3IgPSAi4piFIiBpZiBzY29yZSA+PSBiZXN0X3Njb3JlIGVsc2UgIiAiCiAgICAgICAgICAgIHByaW50KGYiICB7aW5kaWNhdG9yfSBbe2lkeCsxOjJkfS97YXJncy5wb3B1bGF0aW9ufV0gc2NvcmU9e3Njb3JlOi40Zn0iKQoKICAgICAgICAgICAgaWYgc2NvcmUgPiBiZXN0X3Njb3JlOgogICAgICAgICAgICAgICAgYmVzdF9zY29yZSA9IHNjb3JlCiAgICAgICAgICAgICAgICBiZXN0X3RoZXRhID0gdGhldGEuY29weSgpCiAgICAgICAgICAgICAgICAjIFNhdmUgY3VycmVudCBiZXN0IGltbWVkaWF0ZWx5CiAgICAgICAgICAgICAgICBiZXN0X3dlaWdodHMgPSBNTFBUcmFja1BsYW5uZXIudW5wYWNrKGJlc3RfdGhldGEsIGhpZGRlbl9zaXplcykKICAgICAgICAgICAgICAgIG5wLnNhdmV6KHN0cihvdXRwdXRfZGlyIC8gInBsYW5uZXJfd2VpZ2h0cy5ucHoiKSwgKipiZXN0X3dlaWdodHMpCgogICAgICAgIGVzLnRlbGwoY2FuZGlkYXRlcywgc2NvcmVzKQoKICAgICAgICBnZW5fdGltZSA9IHRpbWUudGltZSgpIC0gdF9nZW5fc3RhcnQKICAgICAgICBoaXN0b3J5LmFwcGVuZCh7CiAgICAgICAgICAgICJnZW5lcmF0aW9uIjogZ2VuLAogICAgICAgICAgICAiYmVzdF9zY29yZSI6IGZsb2F0KGJlc3Rfc2NvcmUpLAogICAgICAgICAgICAiZ2VuX21heF9zY29yZSI6IGZsb2F0KHNjb3Jlcy5tYXgoKSksCiAgICAgICAgICAgICJnZW5fbWVhbl9zY29yZSI6IGZsb2F0KHNjb3Jlcy5tZWFuKCkpLAogICAgICAgICAgICAic2lnbWEiOiBmbG9hdChlcy5zaWdtYSksCiAgICAgICAgICAgICJnZW5fdGltZV9zIjogZ2VuX3RpbWUsCiAgICAgICAgfSkKICAgICAgICBwcmludChmIiAgQmVzdCBzbyBmYXI6IHtiZXN0X3Njb3JlOi40Zn0gIHwgIEdlbiBtYXg6IHtzY29yZXMubWF4KCk6LjRmfSAgKHtnZW5fdGltZTouMGZ9cykiKQoKICAgICAgICAjIFNhdmUgaGlzdG9yeSBhZnRlciBlYWNoIGdlbmVyYXRpb24KICAgICAgICAob3V0cHV0X2RpciAvICJzZWFyY2hfaGlzdG9yeS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKGhpc3RvcnksIGluZGVudD0yKSkKCiAgICAjIOKUgOKUgCBGaW5hbCBvdXRwdXQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAjIFNhdmUgYmVzdCB3ZWlnaHRzCiAgICBiZXN0X3dlaWdodHMgPSBNTFBUcmFja1BsYW5uZXIudW5wYWNrKGJlc3RfdGhldGEsIGhpZGRlbl9zaXplcykKICAgIG5wLnNhdmV6KHN0cihvdXRwdXRfZGlyIC8gInBsYW5uZXJfd2VpZ2h0cy5ucHoiKSwgKipiZXN0X3dlaWdodHMpCgogICAgIyBTYXZlIGZpbmFsIHBsYW5uZXIgY29uZmlnCiAgICBmaW5hbF9jZmcgPSB7CiAgICAgICAgInBsYW5uZXJfdHlwZSI6ICJtbHAiLAogICAgICAgICJtbHBfd2VpZ2h0c19wYXRoIjogInBsYW5uZXJfd2VpZ2h0cy5ucHoiLAogICAgICAgICJtbHBfaGlkZGVuIjogaGlkZGVuX3NpemVzLAogICAgICAgICJzdGFuZF9zZWNvbmRzIjogMS4wLAogICAgICAgICJ0cmFpbmluZ19pbmZvIjogewogICAgICAgICAgICAiaXRlcmF0aW9ucyI6IGFyZ3MuaXRlcmF0aW9ucywKICAgICAgICAgICAgInBvcHVsYXRpb24iOiBhcmdzLnBvcHVsYXRpb24sCiAgICAgICAgICAgICJldmFsX3NlY29uZHMiOiBhcmdzLmV2YWxfc2Vjb25kcywKICAgICAgICAgICAgImJlc3RfY29tcG9zaXRlX3Njb3JlIjogZmxvYXQoYmVzdF9zY29yZSksCiAgICAgICAgfSwKICAgIH0KICAgIChvdXRwdXRfZGlyIC8gInBsYW5uZXJfY29uZmlnLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZmluYWxfY2ZnLCBpbmRlbnQ9MikpCgogICAgIyBDbGVhbiB1cCBjYW5kaWRhdGUgZGlyZWN0b3JpZXMKICAgIGlmIHRtcF9yb290LmV4aXN0cygpOgogICAgICAgIHNodXRpbC5ybXRyZWUodG1wX3Jvb3QsIGlnbm9yZV9lcnJvcnM9VHJ1ZSkKCiAgICBwcmludChmIlxueyc9Jyo2MH0iKQogICAgcHJpbnQoZiJUcmFpbmluZyBjb21wbGV0ZS4gIEJlc3QgY29tcG9zaXRlIHNjb3JlOiB7YmVzdF9zY29yZTouNGZ9IikKICAgIHByaW50KGYiV2VpZ2h0cyBzYXZlZCB0bzogICB7b3V0cHV0X2RpciAvICdwbGFubmVyX3dlaWdodHMubnB6J30iKQogICAgcHJpbnQoZiJDb25maWcgc2F2ZWQgdG86ICAgIHtvdXRwdXRfZGlyIC8gJ3BsYW5uZXJfY29uZmlnLmpzb24nfSIpCiAgICBwcmludChmInsnPScqNjB9IikKICAgIHByaW50KCJOZXh0OiBydW4gZnVsbCBldmFsdWF0aW9uIHdpdGg6IikKICAgIHByaW50KGYiICBweXRob24gcnVuX3RyYWNrX2JvbnVzLnB5IFxcIikKICAgIHByaW50KGYiICAgIC0tY2hlY2twb2ludC1kaXIgPHlvdXJfY2hlY2twb2ludD4gXFwiKQogICAgcHJpbnQoZiIgICAgLS1wbGFubmVyLWNvbmZpZyB7b3V0cHV0X2RpciAvICdwbGFubmVyX2NvbmZpZy5qc29uJ30gXFwiKQogICAgcHJpbnQoZiIgICAgLS1jb25maWcgPGNvbmZpZy5qc29uPiBcXCIpCiAgICBwcmludChmIiAgICAtLW91dHB1dC1kaXIgYXJ0aWZhY3RzL3RyYWNrX2V2YWwgXFwiKQogICAgcHJpbnQoZiIgICAgLS1lbnRyeS1uYW1lIDx5b3VyX3RlYW0+IFxcIikKICAgIHByaW50KGYiICAgIC0tcmVuZGVyLWV2ZXJ5IDEwIC0tcmVuZGVyLWZwcyA1IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='
(COURSE_REPO_DIR / 'train_highlevel_mlp.py').write_text(
    _b64.b64decode(_train_b64).decode(), encoding='utf-8')
print('Written train_highlevel_mlp.py')


## 3. Read the assignment requirements

Skim the short specs before editing.


In [ ]:
%cd {COURSE_REPO_DIR}
!sed -n '1,180p' docs/assignment_requirements.md
!sed -n '1,140p' docs/controller_interface.md
!sed -n '1,120p' docs/high_level_optimization_guide.md


## 4. Copy Go2 assets


In [ ]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}


## 5. Inspect the low-level Go2 environment


In [ ]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_2


## 6. Read the important starter files


In [ ]:
!sed -n '1,180p' go2_pg_env/joystick.py
!sed -n '1,220p' go2_pg_env/track.py
!sed -n '1,220p' track_bonus/controller_interface.py
!sed -n '1,220p' track_bonus/planner.py
!sed -n '1,220p' run_track_bonus.py


## 7. Define a Colab-friendly low-level training config

This is a normal Colab training starting point, not a quick test. Reduce the step counts for experiments if needed.


In [ ]:
import json

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)


## 8. Dry-run training config


In [ ]:
!python train.py --config configs/colab_runtime_config.json --dry-run


## 9. Train or reuse a low-level checkpoint

Use a newly trained checkpoint or point `CHECKPOINT_DIR` to a HW1 `best_checkpoint`.


## 9a. Upload HW1 Checkpoint

Upload `best_checkpoint.zip` from your local machine, then unzip it.
Skip this if you trained a new low-level policy in step 9.

In [ ]:
# Upload best_checkpoint.zip from your local machine
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # select best_checkpoint.zip

zip_name = next(k for k in uploaded if k.endswith('.zip'))
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

ckpt = Path('/content/best_checkpoint')
assert ckpt.exists(), f'Extraction failed: {ckpt} not found'
print('Checkpoint ready:', sorted(str(p.name) for p in ckpt.iterdir()))

In [ ]:
# Use HW1 best_checkpoint directly (already trained, composite_score=0.97)
# Upload your HW1 best_checkpoint folder to Colab first, then set the path:
#   Files → Upload → select the entire best_checkpoint/ folder
#   Or use: from google.colab import files; files.upload()

# Option A: HW1 checkpoint uploaded to /content/best_checkpoint
CHECKPOINT_DIR = Path('/content/best_checkpoint')

# Option B: train a new low-level policy from scratch (~40-60 min on A100)
# !python train.py \
#   --config configs/colab_runtime_config.json \
#   --stage both \
#   --output-dir artifacts/low_level_train
# CHECKPOINT_DIR = COURSE_REPO_DIR / 'artifacts' / 'low_level_train' / 'best_checkpoint'

PLANNER_CONFIG = COURSE_REPO_DIR / 'configs' / 'starter_planner.json'
print('checkpoint path:', CHECKPOINT_DIR)
print('checkpoint exists:', CHECKPOINT_DIR.exists())


## 10. Run single-policy track evaluation

The smoke command is short and writes to `track_eval_smoke`. For submission, run the full command into `track_eval`.


In [ ]:
TRACK_EVAL_SMOKE_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval_smoke"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

if CHECKPOINT_DIR.exists():
    print("Found checkpoint. Running short no-render track smoke eval...")
    !python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config {PLANNER_CONFIG} \
      --config configs/colab_runtime_config.json \
      --output-dir {TRACK_EVAL_SMOKE_DIR} \
      --entry-name {TEAM_NAME} \
      --duration-seconds 5 \
      --no-render
else:
    print("Skipping track eval: CHECKPOINT_DIR does not exist yet.")
    print("Train a policy above or set CHECKPOINT_DIR to your HW1 best_checkpoint, then rerun this cell.")

# Full evaluation with video, after the smoke run works:
# !python run_track_bonus.py \
#   --checkpoint-dir {CHECKPOINT_DIR} \
#   --planner-config {PLANNER_CONFIG} \
#   --config configs/colab_runtime_config.json \
#   --output-dir {TRACK_EVAL_DIR} \
#   --entry-name {TEAM_NAME} \
#   --render-every 10 \
#   --render-fps 5


## 11. Train your high-level planner

You are expected to train a learned high-level planner. The command below is only a starter parameter search for debugging the loop; replace the planner internals with your own MLP, RL policy, or other trained policy while keeping the same 5D -> [vx, vy, yaw_rate] interface.

Do not change the track geometry. The evaluator uses the fixed official oval for reset, scoring, and rendering.

A valid learned planner has trained parameters, such as MLP weights. Store those weights in your submission and load them from `StarterTrackPlanner.load(planner_config)`.


In [ ]:
# 11a. Create initial MLP weights and planner config
import json, numpy as np, sys
sys.path.insert(0, str(COURSE_REPO_DIR))
from track_bonus.planner import MLPTrackPlanner

HIDDEN_SIZES = [32, 16]
HIGHLEVEL_MLP_DIR = COURSE_REPO_DIR / 'artifacts' / 'highlevel_mlp'
HIGHLEVEL_MLP_DIR.mkdir(parents=True, exist_ok=True)

init_weights = MLPTrackPlanner.make_weights(HIDDEN_SIZES, seed=42)
np.savez(str(HIGHLEVEL_MLP_DIR / 'planner_weights.npz'), **init_weights)

mlp_cfg = {
    'planner_type': 'mlp',
    'mlp_weights_path': 'planner_weights.npz',
    'mlp_hidden': HIDDEN_SIZES,
    'stand_seconds': 1.0,
}
(HIGHLEVEL_MLP_DIR / 'planner_config.json').write_text(json.dumps(mlp_cfg, indent=2))

n_params = MLPTrackPlanner.param_count(HIDDEN_SIZES)
print(f'MLP: 5->{HIDDEN_SIZES[0]}->{HIDDEN_SIZES[1]}->3  ({n_params} parameters)')
print('Init weights ->', HIGHLEVEL_MLP_DIR / 'planner_weights.npz')


In [ ]:
# 11b. Smoke test: 5-second eval with initial (untrained) MLP weights
MLP_PLANNER_CONFIG = HIGHLEVEL_MLP_DIR / 'planner_config.json'
SMOKE_DIR = HIGHLEVEL_MLP_DIR / '_smoke'
if CHECKPOINT_DIR.exists():
    !python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config {MLP_PLANNER_CONFIG} \
      --config configs/colab_runtime_config.json \
      --output-dir {SMOKE_DIR} \
      --entry-name smoke_mlp \
      --duration-seconds 5 \
      --no-render
    import json
    r = json.loads((SMOKE_DIR / 'results.json').read_text())
    print('Smoke composite score:', r['scores']['composite_score'])
else:
    print('Train the low-level policy in Step 9 first.')


In [ ]:
# 11c. CMA-ES training of MLP high-level planner
# ~30-50 min with --iterations 40 --population 10 --eval-seconds 15
if CHECKPOINT_DIR.exists():
    !python train_highlevel_mlp.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --config configs/colab_runtime_config.json \
      --output-dir {HIGHLEVEL_MLP_DIR} \
      --mlp-hidden 32,16 \
      --iterations 40 \
      --population 10 \
      --eval-seconds 15 \
      --sigma0 0.3 \
      --seed 42
else:
    print('Train the low-level policy in Step 9 first.')


In [ ]:
# 11d. Switch to trained MLP planner and show training summary
PLANNER_CONFIG = HIGHLEVEL_MLP_DIR / 'planner_config.json'
print('PLANNER_CONFIG ->', PLANNER_CONFIG)
import json
if (HIGHLEVEL_MLP_DIR / 'search_history.json').exists():
    hist = json.loads((HIGHLEVEL_MLP_DIR / 'search_history.json').read_text())
    best = max(h['best_score'] for h in hist)
    print(f'Training: {len(hist)} generations, best composite = {best:.4f}')


In [ ]:
# 11e. Full track evaluation with trained MLP planner (produces video)
TRACK_EVAL_DIR = COURSE_REPO_DIR / 'artifacts' / 'track_eval'
if CHECKPOINT_DIR.exists():
    !python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config {PLANNER_CONFIG} \
      --config configs/colab_runtime_config.json \
      --output-dir {TRACK_EVAL_DIR} \
      --entry-name {TEAM_NAME} \
      --render-every 10 \
      --render-fps 5
    import json
    r = json.loads((TRACK_EVAL_DIR / 'results.json').read_text())
    s, m = r['scores'], r['metrics']
    print(f"Composite:  {s['composite_score']:.3f}")
    print(f"Completion: {m['lap_completion']*100:.1f}%  |  Distance: {m['valid_distance_m']:.1f}m")
    print(f"Fall: {m['fall']}  |  Boundary violation: {m['boundary_violation']}")
    if m.get('finish_time'):
        print(f"Finish time: {m['finish_time']:.1f}s")


## 12. Create submission metadata


In [ ]:
import json
submission = {
    "team_name": TEAM_NAME,
    "track2_option": "leaderboard",
    "checkpoint_dir": "best_checkpoint",
    "planner_config": "planner_config.json",
    "planner_code": "track_bonus/planner.py",
    "planner_weights": "planner_weights.npz",
    "high_level_planner_type": "learned_mlp_cmaes",
    "track_eval": "track_eval/results.json",
    "notes": (
        "Low-level: trained from scratch (stage 1 + stage 2, 15M env steps). "
        "High-level: 5->32->16->3 MLP trained with diagonal CMA-ES "
        "(40 generations x 10 population, 15s eval per candidate). "
        "Failed idea: PD parameter tuning via train_highlevel_starter.py -- "
        "converged faster but lower composite score than MLP on corners."
    ),
}
(COURSE_REPO_DIR / 'submission.json').write_text(json.dumps(submission, indent=2))
print((COURSE_REPO_DIR / 'submission.json').read_text())


## 13. Final local checklist

This only checks local paths. If `track_eval/results.json` is missing, run the full evaluation command in Step 10.


In [ ]:
from pathlib import Path
expected = {
    "checkpoint": CHECKPOINT_DIR,
    "planner_config": PLANNER_CONFIG,
    "planner_weights": PLANNER_CONFIG.parent / "planner_weights.npz",
    "submission_json": COURSE_REPO_DIR / "submission.json",
    "track_eval_results": TRACK_EVAL_DIR / "results.json",
}
for label, path in expected.items():
    print(label, path, "OK" if Path(path).exists() else "MISSING")
print("\nSubmit: best_checkpoint/, planner_config.json, planner_weights.npz,",
      "track_bonus/planner.py, train_highlevel_mlp.py, submission.json,",
      "track_eval/results.json, and your short report.")
